In [84]:
import pandas as pd
import requests

In [85]:
visits = pd.read_csv('visits.csv')
registrations = pd.read_csv('registrations.csv')

In [86]:
visits.head()

,uuid,platform,user_agent,date
0,92e2498e-3202-4286-a48e-a4abe5131313,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,2023-06-01T11:12:38
1,92e2498e-3202-4286-a48e-a4abe5131313,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,2023-05-30T13:44:46
2,92e2498e-3202-4286-a48e-a4abe5131313,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,2023-05-29T04:37:54
3,0919c209-9aa2-4c11-ba6f-6a15669954cf,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_2...,2023-06-01T05:58:14
4,0919c209-9aa2-4c11-ba6f-6a15669954cf,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_2...,2023-05-29T01:22:31


In [87]:
registrations.head()

,date,user_id,email,platform,registration_type
0,2023-03-01T07:40:13,2e0f6bb8-b029-4f45-a786-2b53990d37f1,ebyrd@example.org,web,google
1,2023-03-01T13:14:00,f007f97c-9d8b-48b5-af08-119bb8f6d9b6,knightgerald@example.org,web,email
2,2023-03-01T03:05:50,24ff46ae-32b3-4a74-8f27-7cf0b8f32f15,cherylthompson@example.com,web,apple
3,2023-03-01T00:04:47,3e9914e1-5d73-4c23-b25d-b59a3aeb2b60,halldavid@example.org,web,email
4,2023-03-01T18:31:52,27f875fc-f8ce-4aeb-8722-0ecb283d0760,denise86@example.net,web,google


In [88]:
visits = visits[~visits['user_agent'].str.contains('bot', case=False, na=False)].copy()

visits['date'] = pd.to_datetime(visits['date'])
registrations['date'] = pd.to_datetime(registrations['date'])

visits['date_group'] = visits['date'].dt.date
registrations['date_group'] = registrations['date'].dt.date

visits_grouped = (
    visits
    .groupby(['date_group', 'platform'])
    .size()
    .reset_index(name='visits')
)

registrations_grouped = (
    registrations
    .groupby(['date_group', 'platform'])
    .size()
    .reset_index(name='registrations')
)

df = pd.merge(
    visits_grouped,
    registrations_grouped,
    on=['date_group', 'platform'],
    how='outer'
)

df['visits'] = df['visits'].fillna(0)
df['registrations'] = df['registrations'].fillna(0)

df['conversion'] = 0.0
mask = df['visits'] > 0
df.loc[mask, 'conversion'] = (df.loc[mask, 'registrations'] / df.loc[mask, 'visits']) * 100

df = df.sort_values(['date_group', 'platform'])

In [89]:
df.head()

,date_group,platform,visits,registrations,conversion
0,2023-03-01,android,75,61,81.333333
1,2023-03-01,ios,22,18,81.818182
2,2023-03-01,web,844,8,0.947867
3,2023-03-02,android,67,59,88.059701
4,2023-03-02,ios,31,24,77.419355


In [90]:
df = df.sort_values('date_group')
df['date_group'] = df['date_group'].astype(str)
df.to_json('conversion.json')

In [91]:
df

,date_group,platform,visits,registrations,conversion
0,2023-03-01,android,75,61,81.333333
1,2023-03-01,ios,22,18,81.818182
2,2023-03-01,web,844,8,0.947867
3,2023-03-02,android,67,59,88.059701
4,2023-03-02,ios,31,24,77.419355
...,...,...,...,...,...
546,2023-08-30,android,35,27,77.142857
548,2023-08-30,web,1357,34,2.505527
550,2023-08-31,ios,50,36,72.000000
549,2023-08-31,android,57,42,73.684211
